# 03 — Bivariate analysis

Compares descriptive source features by the supplied suspicious-activity label. It does not infer causation.

In [1]:
# Setup
from pathlib import Path
import logging, random
import numpy as np
import pandas as pd
random.seed(42); np.random.seed(42)
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
paths = [p for root in (Path('/kaggle/input'), Path('data/original')) if root.exists() for p in root.rglob('ml_features.csv')]
if not paths: raise FileNotFoundError('Upload the source bundle to Kaggle Input or data/original.')
features = pd.read_csv(paths[0])
label = 'is_suspicious_tx'
assert features[label].isin([0, 1]).all(), 'Unexpected label values'


In [2]:
summary = features.groupby(label).agg(rows=(label, 'size'), amount_mean_npr=('amount_local_npr', 'mean'), amount_median_npr=('amount_local_npr', 'median'), cross_border_rate=('cross_border_flag', 'mean'), currency_mismatch_rate=('currency_mismatch', 'mean'), above_1m_rate=('above_1M_NPR', 'mean'))
summary['share'] = summary['rows'] / len(features)
display(summary)
for field in ['cross_border_flag', 'currency_mismatch', 'above_1M_NPR', 'above_10M_NPR']:
    display(pd.crosstab(features[field], features[label], normalize='columns').rename_axis(index=field, columns=label))
numeric = ['amount_local_npr', 'log_amount', 'sender_country_risk', 'receiver_country_risk', 'velocity_sum_10tx', 'tx_count_10', 'tx_count_30']
display(features[numeric + [label]].corr(numeric_only=True)[[label]].sort_values(label, ascending=False))


,rows,amount_mean_npr,amount_median_npr,cross_border_rate,currency_mismatch_rate,above_1m_rate,share
is_suspicious_tx,,,,,,,
0,99886,1.756721e+06,1236438.750,0.101025,0.116883,0.593807,0.996647
1,336,5.489350e+06,1880738.145,0.163690,0.142857,0.815476,0.003353


is_suspicious_tx,0,1
cross_border_flag,,
0,0.898975,0.83631
1,0.101025,0.16369


is_suspicious_tx,0,1
currency_mismatch,,
0,0.883117,0.857143
1,0.116883,0.142857


is_suspicious_tx,0,1
above_1M_NPR,,
0,0.406193,0.184524
1,0.593807,0.815476


is_suspicious_tx,0,1
above_10M_NPR,,
0,0.992501,0.85119
1,0.007499,0.14881


,is_suspicious_tx
is_suspicious_tx,1.000000
amount_local_npr,0.043106
log_amount,0.027610
receiver_country_risk,0.024502
sender_country_risk,-0.001973
velocity_sum_10tx,-0.009261
tx_count_30,-0.036957
tx_count_10,-0.043611


## Conclusion
The label is highly imbalanced. Preserve this rate in the Phase 2 hand-off and use stratified evaluation later; associations here are descriptive only.